<a href="https://colab.research.google.com/github/kkm1121/ndvia-yolo/blob/20250404/ndvia_yolo_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 📌 필수 라이브러리 설치
!pip install ultralytics yt-dlp opencv-python moviepy ffmpeg-python

import cv2
import torch
import os
import yt_dlp
import ffmpeg
from ultralytics import YOLO
from moviepy.editor import VideoFileClip
from IPython.display import display, Video

# 🚀 1️⃣ 유튜브 영상 다운로드 (480p, 최대한 빠르게)
def download_youtube_video(youtube_url, output_path="video.mp4"):
    ydl_opts = {
        'format': 'best[height<=480]',  # 480p로 다운로드
        'outtmpl': output_path,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])

# 🚗 2️⃣ YOLOv8을 사용해 자동차 감지 & 개수 세기
def detect_cars(video_path, output_path):
    model = YOLO("yolov8n.pt")  # YOLOv8 모델 로드
    cap = cv2.VideoCapture(video_path)

    # 🚀 비디오 저장 설정 (H.264로 인코딩하여 Colab에서 재생 가능하도록 조정)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    car_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame)

        for r in results:
            for box in r.boxes:
                class_id = int(box.cls[0])
                if class_id in [2, 3, 5, 7]:  # 🚗 자동차(2), 트럭(3), 버스(5), 트레일러(7)
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    cv2.putText(frame, "Car", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    car_count += 1

        out.write(frame)  # 저장

    cap.release()
    out.release()
    print(f"✅ 감지된 자동차 개수: {car_count}")

# ✂️ 3️⃣ 30초~35초 구간 자르기
def trim_video(input_path, output_path, start_time, end_time):
    clip = VideoFileClip(input_path).subclip(start_time, end_time)
    clip.write_videofile(output_path, codec="libx264", fps=30)

# 🎬 4️⃣ Colab에서 영상 원활하게 재생
def play_video(video_path):
    os.system(f"ffmpeg -i {video_path} -vcodec libx264 output.mp4 -y")  # 🔥 ffmpeg로 최적화
    display(Video("output.mp4", embed=True))

# 🌐 5️⃣ 실행하기
youtube_url = "https://www.youtube.com/watch?v=QtO6I9tGdbg"  # 🎯 사용할 유튜브 링크

# ① 유튜브 영상 다운로드
download_youtube_video(youtube_url, "video.mp4")

# ② 30초~35초 구간 자르기
trim_video("video.mp4", "trimmed_video.mp4", 30, 35)

# ③ 자동차 감지 & 개수 세기
detect_cars("trimmed_video.mp4", "car_detected.mp4")

# ④ 최적화 후 영상 재생
play_video("car_detected.mp4")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.2/172.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.3/977.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  if event.key is 'enter':



[youtube] Extracting URL: https://www.youtube.com/watch?v=QtO6I9tGdbg
[youtube] QtO6I9tGdbg: Downloading webpage
[youtube] QtO6I9tGdbg: Downloading tv client config
[youtube] QtO6I9tGdbg: Downloading player 73381ccc-main
[youtube] QtO6I9tGdbg: Downloading tv player API JSON
[youtube] QtO6I9tGdbg: Downloading ios player API JSON
[youtube] QtO6I9tGdbg: Downloading m3u8 information
[info] QtO6I9tGdbg: Downloading 1 format(s): 18
[download] Destination: video.mp4
[download] 100% of    6.00MiB in 00:00:00 at 11.55MiB/s  
Moviepy - Building video trimmed_video.mp4.
MoviePy - Writing audio in trimmed_videoTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video trimmed_video.mp4



Moviepy - Done !
Moviepy - video ready trimmed_video.mp4


100%|██████████| 6.25M/6.25M [00:00<00:00, 380MB/s]



0: 384x640 3 persons, 1 car, 1 sandwich, 1 donut, 50.9ms
Speed: 7.9ms preprocess, 50.9ms inference, 334.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 sandwich, 1 donut, 6.9ms
Speed: 2.4ms preprocess, 6.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 sandwich, 1 donut, 7.3ms
Speed: 2.7ms preprocess, 7.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 donut, 7.5ms
Speed: 2.1ms preprocess, 7.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 sandwich, 1 donut, 6.5ms
Speed: 2.1ms preprocess, 6.5ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 2 cars, 1 truck, 1 sandwich, 1 donut, 6.5ms
Speed: 1.9ms preprocess, 6.5ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 car, 1 sandwich, 7.3ms
Speed: 1.9ms preprocess, 7.3ms inference, 1.2ms postp